In [1]:
from dataclasses import dataclass
import csv
from typing import List

@dataclass
class Player:
    player_id: str
    player_name: str
    age: int
    nationality: str
    club: str
    position: str
    overall_rating: int
    potential_rating: int
    matches_played: int
    goals: int
    assists: int
    minutes_played: int
    market_value_million_eur: int
    contract_years_left: int
    injury_prone: str
    transfer_risk_level: str

@dataclass
class PlayerProcessed(Player):
    minutes_for_goal: float
    aportation_cost: float

    def to_dict(self):
        return {
            'player_id': self.player_id,
            'player_name': self.player_name ,
            'age': self.age ,
            'nationality': self.nationality ,
            'club': self.club ,
            'position': self.position ,
            'overall_rating': self.overall_rating ,
            'potential_rating': self.potential_rating ,
            'matches_played': self.matches_played ,
            'goals': self.goals ,
            'assists': self.assists ,
            'minutes_played': self.minutes_played ,
            'market_value_million_eur': self.market_value_million_eur ,
            'contract_years_left': self.contract_years_left ,
            'injury_prone': self.injury_prone ,
            'transfer_risk_level': self.transfer_risk_level,
            'minutes_for_goal': str(self.minutes_for_goal),
            'aportation_worth': str(self.aportation_cost) + 'M'
        }
    
    

In [2]:
def cargar_datos(ruta):
    players = []
    with open(ruta, 'r') as f:
        reader = csv.DictReader(f)
        next(reader)
        for row in reader:
            """
                player_id,
                player_name,
                age,
                nationality,
                club,
                position,
                overall_rating,
                potential_rating,
                matches_played,
                goals,
                assists,
                minutes_played,
                market_value_million_eur,
                contract_years_left,
                injury_prone,
                transfer_risk_level"""
            players.append(
                Player(
                    row['player_id'],
                    row['player_name'],
                    int(row['age']),
                    row['nationality'],
                    row['club'],
                    row['position'],
                    int(row['overall_rating']),
                    int(row['potential_rating']),
                    int(row['matches_played']),
                    int(row['goals']),
                    int(row['assists']),
                    int(row['minutes_played']),
                    float(row['market_value_million_eur']),
                    int(row['contract_years_left']),
                    row['injury_prone'],
                    row['transfer_risk_level'],
                )
            )
    return players


In [3]:
def limpiar_datos(players: List[Player], max_age: int, position: str) -> List[Player]:
    filtered_players = []
    for p in players:
        if p.age <= max_age and p.position == position:
            filtered_players.append(p)
    return filtered_players

In [4]:
def procesar_datos(players: List[Player]) -> List[PlayerProcessed]:
    def minutes_for_goal(player: Player):
        if player.goals <= 0:
            return 0
        return player.minutes_played/player.goals
    def costo_por_aportacion(player: Player):
        aportaciones = player.goals + player.assists
        if aportaciones <= 0:
            return player.market_value_million_eur
        return player.market_value_million_eur / aportaciones
    
    data = []
    for player in players:
        data.append(PlayerProcessed(
            player.player_id,
            player.player_name,
            player.age,
            player.nationality,
            player.club,
            player.position,
            player.overall_rating,
            player.potential_rating,
            player.matches_played,
            player.goals,
            player.assists,
            player.minutes_played,
            player.market_value_million_eur,
            player.contract_years_left,
            player.injury_prone,
            player.transfer_risk_level,
            minutes_for_goal(player),
            costo_por_aportacion(player)
        ))
    return data

In [5]:
def guardar_resultados(datos_procesados, nombre_archivo):
    with open(nombre_archivo, 'w+') as f:
        w = csv.DictWriter(f, [
            'player_id',
            'player_name',
            'age',
            'nationality',
            'club',
            'position',
            'overall_rating',
            'potential_rating',
            'matches_played',
            'goals',
            'assists',
            'minutes_played',
            'market_value_million_eur',
            'contract_years_left',
            'injury_prone',
            'transfer_risk_level',
            'minutes_for_goal',
            'aportation_worth'
                    ])
        w.writeheader()
        for player in datos_procesados:
            w.writerow(player.to_dict())

In [6]:
players = cargar_datos('fifa_player_performance_market_value.csv')
strikers_under_23 = limpiar_datos(players, 23, 'ST')
strikers_worth = procesar_datos(strikers_under_23)
guardar_resultados(strikers_worth, 'resultado.csv')